# AC-Zero — Training (Kaggle)

Runs the config-driven policy/value training pipeline (`aczero train`) with a
wall-clock deadline, then writes a **training summary** (metrics, loss curves,
final checkpoint) to `/kaggle/working` — the notebook's saved output.

**Before you run**
- Settings -> **Internet: On** (needed to `pip install` from GitHub).
- Accelerator: **CPU** is fine — the models are small and self-play fans out
  across CPU cores; a GPU is not required.
- Kaggle sessions are capped at ~12 h. `TIME_BUDGET_HOURS` below stops training
  at an iteration boundary with a safety margin, keeping the last checkpoint and
  the streamed metrics so the summary is always produced.
- Training **seeds each self-play episode from the grown dataset** pulled from the
  Hugging Face bucket (`HF_BUCKET`), so it learns on the guaranteed-solvable
  instances produced by `01_generate_dataset`. Set `HF_DOWNLOAD_ON_START = False`
  to fall back to random scrambles. A private bucket needs an `HF_TOKEN` Kaggle
  secret (Add-ons -> Secrets).

## 1. Configuration

In [ ]:
import os

# --- Repository -------------------------------------------------------------
REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"

# --- Training ---------------------------------------------------------------
BACKEND = "alphazero"  # "alphazero" (PUCT self-play) or "ppo"
RANK = 2
SEED = 0
WORKERS = 0  # self-play worker processes; 0 = all CPU cores

# --- Move set -----------------------------------------------------------
# Which move set self-play actually steps with: "strict-ac" | "universal" (see
# ac_zero.moves.universal.MOVE_SET_NAMES; the same names 01_generate_dataset
# writes via ANNOTATE_MOVESETS). The annotations pulled below (for the
# curriculum's distance-to-origin) should be computed under this same move set.
MOVESET = "strict-ac"

# --- Reward -----------------------------------------------------------------
# How each self-play transition is scored (passed straight to the pipeline):
#   "length_reduction_and_goal" — dense length reduction plus a goal bonus (default)
#   "length_reduction"          — dense length reduction only
#   "sparse_goal"               — a bonus on the goal step and nothing else
REWARD_MODE = "length_reduction_and_goal"

# --- Dataset (seed self-play from the grown Hugging Face dataset) ------------
# Training pulls the grown dataset from the HF bucket and seeds each self-play
# episode from one of its guaranteed-solvable presentations instead of a random
# scramble. Add an `HF_TOKEN` Kaggle secret for a private bucket; set
# HF_DOWNLOAD_ON_START = False to fall back to random scrambles.
HF_BUCKET = "HkHk2Prod/alphaac-data"
DATASET_NAME = f"train_rank{RANK}.groups.json"
# Companion annotation file (distances under MOVESET). DATASET_MAX_DIFFICULTY
# reads it.
ANNOTATIONS_NAME = f"train_rank{RANK}.{MOVESET}.annotations.json"
HF_DOWNLOAD_ON_START = True
DATASET_MAX_DIFFICULTY = None  # cap groups by distance to origin (needs annotations); None = all

# --- Time budget (Kaggle hard-kills at ~12 h) -------------------------------
TIME_BUDGET_HOURS = 11.5  # stop training after this much wall-clock
SAFETY_MARGIN_MIN = 15  # head-room reserved for the summary + plots

# --- Run size ---------------------------------------------------------------
MAX_ITERATIONS = 100_000  # upper bound; the time budget is the real stop
CHECKPOINT_EVERY = 5  # iterations between checkpoints (kept on early stop)
PRINT_EVERY = 10  # print a compact status line every N iterations

# --- Output -----------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
RUN_DIR = f"{OUTPUT_DIR}/run_{BACKEND}_rank{RANK}"
DATASET_PATH = f"{OUTPUT_DIR}/{DATASET_NAME}"
ANNOTATIONS_PATH = f"{OUTPUT_DIR}/{ANNOTATIONS_NAME}"

## 2. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys


# --no-deps keeps Kaggle's pre-built torch (training needs it); add the rest.
def pip_install(*args):
    # Quiet install: capture pip's output and surface it only on real failure.
    # Kaggle's base image ships a broken google-adk dependency, so pip prints a
    # "dependency resolver" ERROR on every run even when our install succeeds;
    # capturing keeps that unrelated noise (and download progress bars) out of
    # the log while still raising if the install itself fails.
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *args],
        capture_output=True,
        text=True,
    )
    if proc.returncode:
        print(proc.stdout, proc.stderr, sep="\n")
        proc.check_returncode()


spec = f"git+{REPO_URL}@{REPO_BRANCH}"
pip_install("--no-deps", spec)
pip_install(
    "numpy>=1.26",
    "pydantic>=2",
    "pyyaml>=6",
    "typer>=0.12",
    "rich>=13",
    "gymnasium>=0.29",
    "matplotlib>=3.8",
    "huggingface_hub>=1.21",
)

import torch

print("Installed ac_zero from", spec, "| torch", torch.__version__)

## 3. Download the training dataset from Hugging Face

In [ ]:
import os

from ac_zero.datasets.hub import download_dataset

# Pick up the HF token from a Kaggle secret if it is not already in the env.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # no HF_TOKEN secret attached to this kernel; fine for a public bucket

if HF_DOWNLOAD_ON_START:
    try:
        pulled = download_dataset(
            DATASET_PATH, remote_name=DATASET_NAME, bucket=HF_BUCKET, missing_ok=True
        )
        if pulled is None:
            print("No dataset in bucket; self-play will use random scrambles.")
            DATASET_PATH = None
            ANNOTATIONS_PATH = None
        else:
            print("Seeding self-play from", pulled)
            apulled = download_dataset(
                ANNOTATIONS_PATH, remote_name=ANNOTATIONS_NAME, bucket=HF_BUCKET, missing_ok=True
            )
            if apulled is None:
                print("No annotations in bucket; max_difficulty curriculum unavailable.")
                ANNOTATIONS_PATH = None
            else:
                print("Loaded annotations from", apulled)
    except Exception as exc:
        print("Bucket download skipped; using random scrambles:", exc)
        DATASET_PATH = None
        ANNOTATIONS_PATH = None
else:
    DATASET_PATH = None
    ANNOTATIONS_PATH = None

## 4. Build the training config

In [ ]:
from ac_zero.training.pipeline_config import TrainingPipelineConfig

# Mirrors configs/experiments/alphazero_rank2_heavy.yaml (and a PPO variant),
# but with `iterations` set high so the wall-clock deadline decides when to stop.
is_az = BACKEND == "alphazero"
base = {
    "rank": RANK,
    "agent": BACKEND,
    "model": "residual_mlp",
    "max_moves": 16,
    "total_length_cap": 128,
    "reward_mode": REWARD_MODE,  # see the Reward section in the config cell
    "moveset": MOVESET,  # see the Move set section in the config cell
    "dataset": {
        "count": 8,
        "depth": 6 if is_az else 3,
        "path": DATASET_PATH,  # seed self-play from the HF group dataset (None = scrambles)
        "annotations": ANNOTATIONS_PATH,  # distances for the max_difficulty curriculum
        "max_difficulty": DATASET_MAX_DIFFICULTY,
    },
    "training": {
        "iterations": MAX_ITERATIONS,
        "episodes_per_iteration": 24 if is_az else 8,
        "optimizer_updates": 32,
        "batch_size": 32 if is_az else 16,
        "replay_capacity": 8192,
        "learning_rate": 0.01 if is_az else 0.02,
        "value_loss_weight": 1.0 if is_az else 0.5,
        "mcts_simulations": 256,
        "c_puct": 1.5,
        "ppo_epochs": 4,
        "ppo_gamma": 0.99,
        "ppo_lambda": 0.95,
        "ppo_clip": 0.2,
        "entropy_coef": 0.01,
        "checkpoint_every": CHECKPOINT_EVERY,
        "workers": WORKERS,
        "run_directory": RUN_DIR,
    },
}
config = TrainingPipelineConfig.from_mapping(base)
config.validate()  # max_difficulty needs dataset.annotations
print(
    "backend:",
    config.agent,
    "| reward:",
    config.reward_mode,
    "| moveset:",
    config.moveset,
    "| seeding:",
    "dataset" if config.dataset_path else "scrambles",
    "| iterations cap:",
    config.iterations,
    "| episodes/iter:",
    config.episodes_per_iteration,
    "| run dir:",
    config.run_directory,
)

## 5. Train with a wall-clock deadline

In [ ]:
import os
import time
from pathlib import Path

from ac_zero.training.callbacks import (
    AsciiGraphLogger,
    CallbackManager,
    JsonlEventLogger,
    TerminalProgressLogger,
)
from ac_zero.training.pipeline import run_training_pipeline

START_TIME = time.monotonic()
DEADLINE = START_TIME + TIME_BUDGET_HOURS * 3600
MARGIN_S = SAFETY_MARGIN_MIN * 60


def elapsed_s():
    return time.monotonic() - START_TIME


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


class DeadlineReached(Exception):
    """Raised from a callback to stop training before the Kaggle kill."""


class DeadlineCallback:
    """Stop at the next event once the budget is nearly spent (fires once).

    The pipeline checkpoints every CHECKPOINT_EVERY iterations and streams every
    event to training_events.jsonl, so the last checkpoint and all metrics survive.
    """

    def __init__(self, deadline, margin):
        self.deadline = deadline
        self.margin = margin
        self._fired = False

    def on_event(self, event):
        if self._fired:
            return
        if time.monotonic() >= self.deadline - self.margin:
            self._fired = True
            raise DeadlineReached(f"stopped at {event.phase} step {event.step}")

    def close(self):
        pass


class ProgressPrinter:
    """Compact console status every PRINT_EVERY iterations (avoids flooding output)."""

    def __init__(self, every):
        self.every = every
        self.loss = {}

    def on_event(self, event):
        m = event.metrics
        for k in ("total_loss", "policy_loss", "value_loss"):
            if k in m:
                self.loss[k] = m[k]
        if event.phase == "self_play" and "iteration" in m:
            it = int(m["iteration"])
            if it == 1 or it % self.every == 0:
                print(
                    f"[{hms(elapsed_s())}] iter {it:>5} | "
                    f"return {m.get('mean_return', float('nan')):+.3f} "
                    f"success {m.get('success_rate', float('nan')):.2f} "
                    f"loss {self.loss.get('total_loss', float('nan')):.4f} "
                    f"| {hms(DEADLINE - time.monotonic())} left"
                )

    def close(self):
        pass


log_dir = Path(RUN_DIR) / "logs"
art_dir = Path(RUN_DIR) / "artifacts"
# File loggers keep the full record; console loggers are muted (a 12 h run emits
# millions of events) in favour of the compact ProgressPrinter below.
devnull = open(os.devnull, "w")
manager = CallbackManager(
    (
        JsonlEventLogger(log_dir / "training_events.jsonl"),
        TerminalProgressLogger(log_dir / "progress.log", stream=devnull),
        AsciiGraphLogger(
            art_dir / "live_graphs.txt",
            art_dir / "final_graphs.txt",
            (
                "total_loss",
                "policy_loss",
                "value_loss",
                "replay_size",
                "episodes",
                "mean_return",
                "success_rate",
            ),
            stream=devnull,
            every_n_events=50,
        ),
        ProgressPrinter(PRINT_EVERY),
        DeadlineCallback(DEADLINE, MARGIN_S),
    )
)

completed = False
summary = None
print(f"Training up to {TIME_BUDGET_HOURS} h (stop with {SAFETY_MARGIN_MIN} min to spare)...")
try:
    summary = run_training_pipeline(config, SEED, callbacks=manager)
    completed = True
    print(
        f"[{hms(elapsed_s())}] training completed: {summary.iterations} iterations, "
        f"{summary.optimizer_updates} optimizer updates."
    )
except DeadlineReached as stop:
    print(f"[{hms(elapsed_s())}] {stop}; last checkpoint and metrics preserved.")

## 6. Training summary & output artifacts

In [ ]:
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

run = Path(RUN_DIR)

events = []
ev_path = run / "logs" / "training_events.jsonl"
if ev_path.exists():
    for line in ev_path.read_text().splitlines():
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            pass


def series(phase, key):
    out = []
    for e in events:
        if e.get("phase") == phase and key in e.get("metrics", {}):
            out.append(e["metrics"][key])
    return out


steps = series("optimizer", "optimizer_step")
total_loss = series("optimizer", "total_loss")
policy_loss = series("optimizer", "policy_loss")
value_loss = series("optimizer", "value_loss")
iters = series("self_play", "iteration")
mean_return = series("self_play", "mean_return")
success_rate = series("self_play", "success_rate")

ckpt = {}
ckpt_path = run / "checkpoints" / "latest.json"
if ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text())

pipeline_summary = {}
ts_path = run / "artifacts" / "training_summary.json"
if ts_path.exists():
    pipeline_summary = json.loads(ts_path.read_text())

report = {
    "backend": BACKEND,
    "status": "completed" if (completed and pipeline_summary) else "stopped_at_time_budget",
    "wall_clock": hms(elapsed_s()),
    "iterations_completed": max(iters) if iters else ckpt.get("iteration", 0),
    "optimizer_updates": max(steps) if steps else ckpt.get("optimizer_state", {}).get("step", 0),
    "final_total_loss": total_loss[-1] if total_loss else None,
    "final_policy_loss": policy_loss[-1] if policy_loss else None,
    "final_value_loss": value_loss[-1] if value_loss else None,
    "best_success_rate": max(success_rate) if success_rate else None,
    "final_success_rate": success_rate[-1] if success_rate else None,
    "final_mean_return": mean_return[-1] if mean_return else None,
    "checkpoint": str(ckpt_path) if ckpt_path.exists() else None,
    "run_directory": str(run),
}
if pipeline_summary:
    report["certificate_verified"] = pipeline_summary.get("certificate_verified")
    report["replay_size"] = pipeline_summary.get("replay_size")

(Path(OUTPUT_DIR) / "training_report.json").write_text(json.dumps(report, indent=2))


def save_plot(x, ys, labels, xlabel, ylabel, title, fname):
    ys = [y for y in ys if y]
    if not x or not ys:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    for y, lab in zip(ys, labels):
        ax.plot(x[: len(y)], y[: len(x)], label=lab)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if len(labels) > 1:
        ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    path = Path(OUTPUT_DIR) / fname
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return path


plots = [
    save_plot(
        steps, [total_loss], ["total"], "optimizer step", "loss", "Total loss", "loss_total.png"
    ),
    save_plot(
        steps,
        [policy_loss, value_loss],
        ["policy", "value"],
        "optimizer step",
        "loss",
        "Policy & value loss",
        "loss_components.png",
    ),
    save_plot(
        iters,
        [success_rate, mean_return],
        ["success rate", "mean return"],
        "iteration",
        "value",
        "Self-play success & return",
        "selfplay.png",
    ),
]

lines = [
    f"# Training summary — {BACKEND} (rank 2)",
    "",
    f"- Status: **{report['status']}**",
    f"- Wall-clock: {report['wall_clock']}",
    f"- Iterations completed: {report['iterations_completed']}",
    f"- Optimizer updates: {report['optimizer_updates']}",
    f"- Final total loss: {report['final_total_loss']}",
    f"- Final policy / value loss: {report['final_policy_loss']} / {report['final_value_loss']}",
    f"- Best / final success rate: {report['best_success_rate']} / {report['final_success_rate']}",
    f"- Final mean return: {report['final_mean_return']}",
    f"- Checkpoint: {report['checkpoint']}",
]
if "certificate_verified" in report:
    lines.append(f"- Certificate verified: {report['certificate_verified']}")
lines.append("")
(Path(OUTPUT_DIR) / "training_report.md").write_text("\n".join(lines) + "\n")

display(Markdown("\n".join(lines)))
for path in plots:
    if path is not None:
        display(Image(str(path)))

print("Output artifacts in", OUTPUT_DIR, ":")
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    kind = "dir" if p.is_dir() else f"{p.stat().st_size / 1e3:.1f} KB"
    print(f"  {p.name:28s} {kind}")